Markdown
# 📊 Tutorial: Cross-Domain Data Drift & Profile Comparison

### What is Data Drift?
Data Drift occurs when the statistical properties of your input data change over time. In production machine learning pipelines or data quality automation, tracking drift ensures that downstream models don't fail silently due to shifting distributions.

---

### 🧠 Core Concepts Addressed in this Workflow:

1. **Tabular Feature Shifts:** Identifying changes in basic numeric and text columns.
2. **Missing Token Anomalies:** Catching unexpected structural spikes in empty/null values.
3. **Profile Extraction:** Using Arnio's high-performance data profile summaries to run quick comparisons without maintaining copies of heavy datasets.

---

In [ ]:
import pandas as pd
import arnio as ar

# 1. Creating baseline and current ArFrame datasets
baseline_data = {
    "patient_id": [101, 102, 103, 104, 105],
    "blood_pressure": [120.0, 122.0, 118.0, 121.0, 119.0],
    "missing_rate_trigger": [0.01, 0.02, 0.01, 0.03, 0.02]
}

current_data = {
    "patient_id": [201, 202, 203, 204, 205],
    "blood_pressure": [72.0, 68.0, 75.0, 70.0, 71.0],
    "missing_rate_trigger": [0.45, 0.52, None, 0.38, 0.41]
}

baseline_frame = ar.from_pandas(pd.DataFrame(baseline_data))
current_frame = ar.from_pandas(pd.DataFrame(current_data))

# 2. Generating profiles with ar.profile(...)
baseline_profile = ar.profile(baseline_frame)
current_profile = ar.profile(current_frame)

# 3. Comparing them with ar.compare_profiles(...)
comparison_report = ar.compare_profiles(baseline_profile, current_profile)

# 4. Checking CI-style gates by passing both profiles directly
gate_result = ar.check_quality_gates(
    baseline_profile,
    current_profile,
    max_null_ratio_delta=0.05,
    max_numeric_mean_delta_ratio=0.10
)

# 5. Inspecting QualityGateResult.issues directly
print("--- Automated Data Drift Integrity Report ---")
if gate_result.issues:
    print("🚨 ALERT: Production Data Drift Detected!")
    print(gate_result.issues)
else:
    print("✅ SUCCESS: Data metrics are statistically stable.")